In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
movies = pd.read_csv('/content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM/movies.csv')

In [4]:
movies.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


In [6]:
movies.describe()

,id,popularity,vote_average,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,161243.505000,34.697267,6.621150,1547.309400
std,211422.046043,211.684175,0.766231,2648.295789
min,5.000000,0.600000,4.600000,200.000000
25%,10127.750000,9.154750,6.100000,315.000000
50%,30002.500000,13.637500,6.600000,583.500000
75%,310133.500000,25.651250,7.200000,1460.000000
max,934761.000000,10436.917000,8.700000,31917.000000


In [7]:
movies.isnull().sum()

,0
id,0
title,0
genre,3
original_language,0
overview,13
popularity,0
release_date,0
vote_average,0
vote_count,0


In [8]:
movies.columns

Index(['id', 'title', 'genre', 'original_language', 'overview', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Feature Selection Part

In [9]:
movies=movies[['id', 'title', 'genre', 'overview' ]]
movies

,id,title,genre,overview
0,278,The Shawshank Redemption,"Drama,Crime",Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Drama,Crime","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,"Drama,History,War",The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,"Drama,Crime",In the continuing saga of the Corleone crime f...
...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"Action,Science Fiction,War","During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...


In [10]:
movies['tags'] = movies['overview'] + movies['genre']

In [11]:
movies

,id,title,genre,overview,tags
0,278,The Shawshank Redemption,"Drama,Crime",Framed in the 1940s for the double murder of h...,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second...","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Drama,Crime","Spanning the years 1945 to 1955, a chronicle o...","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,"Drama,History,War",The true story of how businessman Oskar Schind...,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,"Drama,Crime",In the continuing saga of the Corleone crime f...,In the continuing saga of the Corleone crime f...
...,...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo...","The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"Action,Science Fiction,War","During World War II, a brave, patriotic Americ...","During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...,A man named Farmer sets out to rescue his kidn...


In [12]:
new_data = movies.drop(columns=['overview', 'genre'])
new_data

,id,title,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...
...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...


In [13]:
cv=CountVectorizer(max_features=1000, stop_words='english')
cv

CountVectorizer(max_features=1000, stop_words='english')

In [14]:
vector=cv.fit_transform(new_data['tags'].values.astype('U')).toarray()

In [15]:
vector.shape

(10000, 1000)

In [16]:
similarity=cosine_similarity(vector)
similarity

array([[1.        , 0.10114435, 0.20851441, ..., 0.11616046, 0.16718346,
        0.09325048],
       [0.10114435, 1.        , 0.14552138, ..., 0.        , 0.06482037,
        0.        ],
       [0.20851441, 0.14552138, 1.        , ..., 0.03713907, 0.1069045 ,
        0.13416408],
       ...,
       [0.11616046, 0.        , 0.03713907, ..., 1.        , 0.04962917,
        0.04152274],
       [0.16718346, 0.06482037, 0.1069045 , ..., 0.04962917, 1.        ,
        0.05976143],
       [0.09325048, 0.        , 0.13416408, ..., 0.04152274, 0.05976143,
        1.        ]])

In [17]:
new_data[new_data['title']=='The Godfather'].index[0]

np.int64(2)

In [19]:
distance=sorted(list(enumerate(similarity[2])), key=lambda vector:vector[1], reverse=True)
for i in distance[0:5]:
    print(new_data.iloc[i[0]].title)

The Godfather
The Godfather: Part II
House of Gucci
Batman: The Killing Joke
Bomb City


In [20]:
def recommend(movie):
    movie_index=new_data[new_data['title']==movie].index[0]
    distance=sorted(list(enumerate(similarity[movie_index])), key=lambda vector:vector[1], reverse=True)
    for i in distance[0:5]:
        print(new_data.iloc[i[0]].title)

In [21]:
recommend('Iron Man')

Iron Man
Guardians of the Galaxy Vol. 2
Star Wars: Episode III - Revenge of the Sith
G.O.R.A.
Mad Max Beyond Thunderdome


In [ ]:
# pickle.dump(new_data, open('movies_list.pkl', 'wb'))
# pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [22]:


# 1. Define the directory path (Folder only)
drive_folder = '/content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM'

# 2. Create the folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"Created new directory: {drive_folder}")

# 3. Define the full file path
file_path = os.path.join(drive_folder, 'movies_list.pkl')

# 4. Save the model
try:
    with open(file_path, 'wb') as f:
        pickle.dump(new_data, f)
    print(f"Success! Model saved to: {file_path}")
except NameError:
    print("Error: The variable 'new_data' is not defined. Make sure you ran the cell where you created your dataframe/list.")
except Exception as e:
    print(f"An error occurred: {e}")

Success! Model saved to: /content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM/movies_list.pkl


In [23]:
# 1. Define the directory path (Folder only)
drive_folder = '/content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM'

# 2. Create the folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"Created new directory: {drive_folder}")

# 3. Define the full file path
file_path = os.path.join(drive_folder, 'similarity.pkl')

# 4. Save the model
try:
    with open(file_path, 'wb') as f:
        pickle.dump(similarity, f)
    print(f"Success! Model saved to: {file_path}")
except NameError:
    print("Error: The variable 'similarity' is not defined. Make sure you ran the cell where you created your similarity matrix.")
except Exception as e:
    print(f"An error occurred: {e}")

Success! Model saved to: /content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM/similarity.pkl


In [24]:
import pickle
import os

# 1. Define the directory path
drive_folder = '/content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM'

# 2. Create the folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

# 3. Define the path for the COMPACT model
# We call it 'similarity_compact.pkl' to distinguish it from the huge 800MB file
file_path = os.path.join(drive_folder, 'similarity_compact.pkl')

# 4. Create and Save the Compact Model
try:
    # Create a dictionary of the top 10 recommended indices for every movie
    similarity_compact = {}
    for i in range(len(similarity)):
        # Storing only the index and similarity score for the top 10 matches
        distances = sorted(list(enumerate(similarity[i])), reverse=True, key=lambda x: x[1])[1:11]
        similarity_compact[i] = distances

    with open(file_path, 'wb') as f:
        pickle.dump(similarity_compact, f)
    print(f"Success! Compact model saved to: {file_path}")
    print("This file is now small enough for GitHub and Streamlit Cloud.")

except NameError:
    print("Error: The variable 'similarity' is not defined.")
except Exception as e:
    print(f"An error occurred: {e}")

Success! Compact model saved to: /content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM/similarity_compact.pkl
This file is now small enough for GitHub and Streamlit Cloud.


In [25]:
# 1. Define the directory path (Folder only)
drive_folder = '/content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM'

# 2. Create the folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"Created new directory: {drive_folder}")

# 3. Define the full file path
file_path = os.path.join(drive_folder, 'similarity.pkl')

# 4. Save the model
try:
    with open(file_path, 'wb') as f:
        pickle.dump(similarity, f)
    print(f"Success! Model saved to: {file_path}")
except NameError:
    print("Error: The variable 'similarity' is not defined. Make sure you ran the cell where you created your similarity matrix.")
except Exception as e:
    print(f"An error occurred: {e}")

Success! Model saved to: /content/drive/MyDrive/Project/MOVIE_RECOM_SYSTEM/similarity.pkl
